# MRS Interactive Corridor Builder (v6.4)

This version addresses your feedback on v6.3:

## ✅ Changes

1) **Map default height = 720px** (instead of 85vh)

2) **Commit / Add Highlight button is back**
- Preview (dashed) → Commit (solid highlight layer)

3) **Scrolling only where it matters**
- The panel stays compact
- The **log/preview output box** is scrollable (and taller)

4) **Better intersection noding (don’t drop unassigned segments)**
Your case still produced a long path even in Allow All. A common cause is that after noding we previously **dropped** segments that could not be confidently assigned a `CodigoFerr`, which can remove critical connector pieces.

In v6.4:
- We keep noded segments even when `CodigoFerr` cannot be assigned (they become `NA`).
- In **Allow all**, NA segments are still allowed.
- In **Prefer MRS**, NA segments are treated as *non‑MRS* for penalty purposes.

5) **Extra diagnostics**
- Print whether any NA-coded segments are being used.

Created: 2026-02-11


In [1]:
# If you already have these installed in THIS kernel environment, leave this cell commented.
#
# !pip -q install geopandas shapely pyproj networkx scipy ipyleaflet ipywidgets pandas


In [2]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx

from shapely.geometry import LineString
from shapely.ops import unary_union, snap

from scipy.spatial import cKDTree
from pyproj import Transformer

from ipyleaflet import (
    Map, GeoData, LayersControl, WidgetControl,
    Polyline, Marker, basemaps,
    FullScreenControl
)
import ipywidgets as widgets


## 1) Configuration

In [3]:
RAILS_SHP_PATH = Path(r"C:\Users\matheus.deoliveira\OneDrive - Wabtec Corporation\Matheus\wabtec\Advanced Technology\MRS\Malha Ferroviária Federal (shp)\Malha Ferroviária Federal (shp)\Estacoes Code\LInhas\Linhas_BR.shp")
STATIONS_SHP_PATH = Path(r"C:\Users\matheus.deoliveira\OneDrive - Wabtec Corporation\Matheus\wabtec\Advanced Technology\MRS\Malha Ferroviária Federal (shp)\Malha Ferroviária Federal (shp)\Estacoes Code\Estacoes\Estacoes.shp")

FILTER_COLUMN = "CodigoFerr"
MRS_CODE = 11

STATION_NAME_FIELD = "NomeEstaca"
STATION_ID_FIELD = None

SOURCE_CRS_IF_MISSING = "EPSG:4674"  # only used if .crs is None
STATION_MAX_DISTANCE_TO_RAIL_M = 8000

STRICT_MRS = "Strict MRS (11 only)"
PREFER_MRS = "Prefer MRS (penalize non-11)"
ALLOW_ALL  = "Allow all (shortest distance)"
DEFAULT_ROUTING_MODE = ALLOW_ALL

NON_MRS_PENALTY_DEFAULT = 10.0
AUTO_INCLUDE_CODIGOFERR_NEAR_SELECTED_POINTS = True
NEAR_POINT_SEARCH_RADIUS_M = 1500

# Topology repair
ROUND_M = 10
SNAP_ENDPOINTS = True
SNAP_TOLERANCE_M = 25

# Intersection noding
NODE_INTERSECTIONS_DEFAULT = True
ASSIGN_CODIGO_MAX_DIST_M = 10.0  # increased from 2m to avoid dropping legit connectors

# Map styles
BASE_RAIL_COLOR = "#2ca02c"
BASE_RAIL_WEIGHT = 3

PREVIEW_COLOR = "#1f77b4"
PREVIEW_WEIGHT = 4
PREVIEW_OPACITY = 0.9
PREVIEW_DASH = '6,6'

HIGHLIGHT_WEIGHT = 5
HIGHLIGHT_OPACITY = 1.0
HIGHLIGHT_LINE_CAP = 'butt'
HIGHLIGHT_LINE_JOIN = 'miter'

MAX_SNAP_TO_GRAPH_M = 2000

# Default map height
MAP_HEIGHT_PX = 720

assert RAILS_SHP_PATH.exists(), f"Rails not found: {RAILS_SHP_PATH}"
assert STATIONS_SHP_PATH.exists(), f"Stations not found: {STATIONS_SHP_PATH}"


# 2) Load data + dropdown

In [4]:
rails_all = gpd.read_file(RAILS_SHP_PATH)
stations_all = gpd.read_file(STATIONS_SHP_PATH)

if rails_all.crs is None and SOURCE_CRS_IF_MISSING:
    rails_all = rails_all.set_crs(SOURCE_CRS_IF_MISSING)
if stations_all.crs is None and SOURCE_CRS_IF_MISSING:
    stations_all = stations_all.set_crs(SOURCE_CRS_IF_MISSING)

print('Rails CRS:', rails_all.crs)
print('Stations CRS:', stations_all.crs)

rails_all[FILTER_COLUMN] = pd.to_numeric(rails_all[FILTER_COLUMN], errors='coerce').round().astype('Int64')
rails_all_m = rails_all.to_crs(epsg=3857)

rails_mrs = rails_all[rails_all[FILTER_COLUMN] == MRS_CODE].copy()
rails_mrs_ll = rails_mrs.to_crs(epsg=4326)
rails_mrs_m = rails_mrs.to_crs(epsg=3857)

stations_m = stations_all.to_crs(epsg=3857)
stations_pts_m = stations_m[stations_m.geometry.type == 'Point'].copy()

rail_union_m = rails_mrs_m.geometry.union_all()
stations_pts_m['dist_to_mrs_m'] = stations_pts_m.geometry.distance(rail_union_m)
stations_near = stations_pts_m[stations_pts_m['dist_to_mrs_m'] <= STATION_MAX_DISTANCE_TO_RAIL_M].copy()

names = stations_near[STATION_NAME_FIELD].astype(str).fillna('')
if STATION_ID_FIELD and STATION_ID_FIELD in stations_near.columns:
    ids = stations_near[STATION_ID_FIELD].astype(str)
    stations_near['station_label'] = names + ' [' + ids + ']'
else:
    dup = names.duplicated(keep=False)
    stations_near['station_label'] = names
    stations_near.loc[dup, 'station_label'] = names[dup] + ' (idx=' + stations_near.loc[dup].index.astype(str) + ')'

stations_near_ll = stations_near.to_crs(epsg=4326)
stations_lookup = stations_near.set_index('station_label')
labels = sorted(list(stations_lookup.index))

print('Stations in dropdown:', len(labels))


Rails CRS: EPSG:4674
Stations CRS: EPSG:4674
Stations in dropdown: 311


## 3) Graph builder with intersection noding (keep NA-coded segments)

Key v6.4 change:
- After noding, we DO NOT drop segments that couldn't be assigned a CodigoFerr.
- Those segments keep CodigoFerr = NA.
- In Allow All, NA edges are allowed.
- In Prefer MRS, NA edges are treated as non-MRS (penalized) so MRS is still preferred.


In [5]:
to_4326 = Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)


def infer_codes_near_points(points_xy, radius_m=NEAR_POINT_SEARCH_RADIUS_M):
    codes = set()
    for (x, y) in points_xy:
        minx, miny = x - radius_m, y - radius_m
        maxx, maxy = x + radius_m, y + radius_m
        subset = rails_all_m.cx[minx:maxx, miny:maxy]
        if len(subset) == 0:
            continue
        pt = gpd.points_from_xy([x], [y])[0]
        d = subset.distance(pt)
        near = subset[d <= radius_m]
        codes.update(near[FILTER_COLUMN].dropna().unique().tolist())
    return sorted(list(codes))


def node_and_assign_codigo_keep_na(rails_sub_m: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    merged = unary_union(list(rails_sub_m.geometry.values))
    noded = gpd.GeoSeries([merged], crs=rails_sub_m.crs).explode(index_parts=False)
    segs = gpd.GeoDataFrame(geometry=noded).reset_index(drop=True)
    segs = segs[segs.geometry.notnull()]
    segs = segs[segs.geometry.type == 'LineString'].copy()

    sidx = rails_sub_m.sindex
    codigos = []

    for geom in segs.geometry.values:
        cand_idx = list(sidx.intersection(geom.bounds))
        if not cand_idx:
            codigos.append(pd.NA)
            continue
        cand = rails_sub_m.iloc[cand_idx]
        dists = cand.distance(geom)
        j = int(dists.idxmin())
        min_dist = float(dists.loc[j])
        if min_dist <= ASSIGN_CODIGO_MAX_DIST_M:
            codigos.append(cand.loc[j, FILTER_COLUMN])
        else:
            codigos.append(pd.NA)

    segs[FILTER_COLUMN] = pd.array(codigos, dtype='Int64')
    return segs


def build_graph(points_xy, routing_mode, penalty_mult, node_intersections=True, progress=None, status=None):
    def set_prog(v, msg=None):
        if progress is not None:
            progress.value = int(v)
        if status is not None and msg is not None:
            status.value = msg

    set_prog(5, 'Selecting rails...')

    # Start from a bbox around all points
    xs = [p[0] for p in points_xy]
    ys = [p[1] for p in points_xy]
    margin = 200_000
    minx, miny = min(xs) - margin, min(ys) - margin
    maxx, maxy = max(xs) + margin, max(ys) + margin

    rails_bbox = rails_all_m.cx[minx:maxx, miny:maxy]

    if routing_mode == STRICT_MRS:
        rails = rails_bbox[rails_bbox[FILTER_COLUMN] == MRS_CODE].copy()
        codes_used = [MRS_CODE]
    elif routing_mode == ALLOW_ALL:
        rails = rails_bbox.copy()
        codes_used = sorted(rails[FILTER_COLUMN].dropna().unique().tolist())
    else:
        # Prefer MRS
        if AUTO_INCLUDE_CODIGOFERR_NEAR_SELECTED_POINTS:
            codes_used = infer_codes_near_points(points_xy)
            if MRS_CODE not in codes_used:
                codes_used = [MRS_CODE] + codes_used
        else:
            codes_used = [MRS_CODE]
        rails = rails_bbox[rails_bbox[FILTER_COLUMN].isin(codes_used)].copy()

    set_prog(20, f'Rails in bbox: {len(rails)} | explode...')

    rails = rails.explode(index_parts=False).reset_index(drop=True)
    rails = rails[rails.geometry.notnull()]
    rails = rails[rails.geometry.type == 'LineString'].copy()

    if len(rails) == 0:
        set_prog(100, 'No rails found in bbox.')
        return nx.Graph(), [], None, {}, codes_used

    if SNAP_ENDPOINTS:
        set_prog(30, 'Snapping linework...')
        u = unary_union(list(rails.geometry.values))
        rails['geometry'] = rails.geometry.apply(lambda g: snap(g, u, SNAP_TOLERANCE_M))

    if node_intersections:
        set_prog(45, 'Noding intersections + assigning CodigoFerr (keeping NA)...')
        rails = node_and_assign_codigo_keep_na(rails)
        na_cnt = int(rails[FILTER_COLUMN].isna().sum())
        set_prog(55, f'Noded segments: {len(rails)} | NA CodigoFerr: {na_cnt}')

    set_prog(65, 'Building graph...')

    def round_xy(xy, r=ROUND_M):
        return (round(xy[0]/r)*r, round(xy[1]/r)*r)

    node_index = {}
    node_coords = []

    def get_node_id(xy):
        xy = round_xy(xy)
        if xy not in node_index:
            node_index[xy] = len(node_coords)
            node_coords.append(xy)
        return node_index[xy]

    G = nx.Graph()

    for _, row in rails.iterrows():
        codigo = row.get(FILTER_COLUMN)  # may be NA
        coords = list(row.geometry.coords)
        for a, b in zip(coords[:-1], coords[1:]):
            u_id = get_node_id(a)
            v_id = get_node_id(b)
            length_m = LineString([a, b]).length

            # Penalize non-MRS (including NA) in Prefer mode
            if routing_mode == PREFER_MRS and (codigo != MRS_CODE):
                w = length_m * float(penalty_mult)
            else:
                w = length_m

            if G.has_edge(u_id, v_id):
                if w < G[u_id][v_id]['weight']:
                    G[u_id][v_id].update({'weight': w, 'length_m': length_m, 'codigo': codigo})
            else:
                G.add_edge(u_id, v_id, weight=w, length_m=length_m, codigo=codigo)

    node_xy = np.array(node_coords)
    tree = cKDTree(node_xy) if len(node_xy) else None

    comps = list(nx.connected_components(G))
    node_to_comp = {}
    for i, comp in enumerate(comps):
        for n in comp:
            node_to_comp[n] = i

    set_prog(85, f'Graph ready: {G.number_of_nodes()} nodes / {G.number_of_edges()} edges')
    return G, node_coords, tree, node_to_comp, codes_used


def stitch_paths(paths):
    if not paths:
        return []
    out = list(paths[0])
    for p in paths[1:]:
        out.extend(p[1:])
    return out


def nodes_to_latlon(node_coords, path_nodes):
    out = []
    for n in path_nodes:
        x, y = node_coords[n]
        lon, lat = to_4326.transform(x, y)
        out.append((lat, lon))
    return out


def path_length_m(G, path_nodes):
    if path_nodes is None:
        return None
    return sum(float(G[u][v].get('length_m', 0.0)) for u, v in zip(path_nodes[:-1], path_nodes[1:]))


def codigo_breakdown_km(G, path_nodes):
    by = {}
    for u, v in zip(path_nodes[:-1], path_nodes[1:]):
        d = G[u][v]
        codigo = d.get('codigo')
        by[codigo] = by.get(codigo, 0.0) + float(d.get('length_m', 0.0)) / 1000.0
    return dict(sorted(by.items(), key=lambda kv: (-kv[1], str(kv[0]))))


## 4) UI — Preview + Commit Highlight (panel compact, output scrollable)

- Map height default = 720px
- Panel stays compact
- Output log is scrollable


In [ ]:
# Map
centroid = rails_mrs_ll.geometry.union_all().centroid
m = Map(center=(float(centroid.y), float(centroid.x)), zoom=6, basemap=basemaps.CartoDB.Positron)

m.layout.height = f'{MAP_HEIGHT_PX}px'
m.layout.width = '100%'
m.add_control(FullScreenControl(position='topleft'))

m.add_layer(GeoData(
    geo_dataframe=rails_mrs_ll[['geometry']],
    name='MRS Rails (CodigoFerr=11)',
    style={'color': BASE_RAIL_COLOR, 'weight': BASE_RAIL_WEIGHT, 'opacity': 0.9}
))

m.add_control(LayersControl(position='topright'))

# Controls
start_dd = widgets.Dropdown(options=labels, description='Station A:', layout=widgets.Layout(width='330px'))
end_dd   = widgets.Dropdown(options=labels, description='Station B:', layout=widgets.Layout(width='330px'))

via_pick_dd = widgets.Dropdown(options=labels, description='Add VIA:', layout=widgets.Layout(width='330px'))
btn_add_via = widgets.Button(description='Add VIA', button_style='info')

via_list = widgets.Select(options=[], description='VIA list:', rows=4, layout=widgets.Layout(width='330px'))
btn_remove_via = widgets.Button(description='Remove', button_style='danger')
btn_up = widgets.Button(description='Up')
btn_down = widgets.Button(description='Down')
btn_clear_vias = widgets.Button(description='Clear', button_style='warning')

routing_mode_dd = widgets.Dropdown(options=[STRICT_MRS, PREFER_MRS, ALLOW_ALL], value=DEFAULT_ROUTING_MODE,
                                   description='Mode:', layout=widgets.Layout(width='330px'))
penalty_slider = widgets.FloatLogSlider(value=NON_MRS_PENALTY_DEFAULT, base=10, min=0, max=2, step=0.1,
                                        description='Penalty:', readout_format='.1f', layout=widgets.Layout(width='330px'))

node_cb = widgets.Checkbox(value=NODE_INTERSECTIONS_DEFAULT, description='Node intersections')

color_picker = widgets.ColorPicker(value='#ff7f0e', description='Color:')
highlight_name = widgets.Text(value='Corridor highlight', description='Name:', layout=widgets.Layout(width='330px'))

btn_preview = widgets.Button(description='Compute Preview', button_style='primary')
btn_commit = widgets.Button(description='Commit Highlight', button_style='success')
btn_explain = widgets.Button(description='Explain', button_style='info')

# Output: scrollable and taller
out = widgets.Output(layout={'border': '1px solid #444', 'padding': '6px', 'max_height': '260px', 'overflow': 'auto'})
progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='info', layout=widgets.Layout(width='98%'))
status = widgets.HTML(value='Ready.')

panel = widgets.VBox([
    widgets.HBox([start_dd, end_dd]),
    widgets.HBox([via_pick_dd, btn_add_via, btn_remove_via, btn_up, btn_down, btn_clear_vias]),
    via_list,
    widgets.HBox([routing_mode_dd, penalty_slider]),
    widgets.HBox([node_cb]),
    widgets.HBox([color_picker, highlight_name]),
    widgets.HBox([btn_preview, btn_commit, btn_explain]),
    progress,
    status,
    out
])

# Panel itself small; only out scrolls
panel.layout.max_height = '320px'

accordion = widgets.Accordion(children=[panel])
accordion.set_title(0, 'Route builder')
accordion.selected_index = 0

toggle_panel = widgets.ToggleButton(value=False, description='Hide panel', icon='window-minimize')

def on_toggle(_):
    if toggle_panel.value:
        accordion.layout.display = 'none'
        toggle_panel.description = 'Show panel'
        toggle_panel.icon = 'window-restore'
    else:
        accordion.layout.display = 'block'
        toggle_panel.description = 'Hide panel'
        toggle_panel.icon = 'window-minimize'

toggle_panel.observe(on_toggle, names='value')

m.add_control(WidgetControl(widget=widgets.VBox([toggle_panel, accordion]), position='topleft'))

# State
preview_layer = None
preview_path_nodes = None
last_G = None
last_node_coords = None
last_seg_diag = None
last_waypoints = None

highlight_layers = {}
highlight_order = []
markers = []


def refresh_highlight_list():
    # (kept simple: rely on LayersControl for toggling; we track internally for removal later if needed)
    pass


def clear_markers():
    for mk in list(markers):
        try:
            m.remove_layer(mk)
        except Exception:
            pass
    markers.clear()


def add_marker(label):
    row = stations_near_ll[stations_near_ll['station_label'] == label].iloc[0]
    mk = Marker(location=(float(row.geometry.y), float(row.geometry.x)), title=label)
    markers.append(mk)
    m.add_layer(mk)


def add_via(_):
    cur = list(via_list.options)
    v = via_pick_dd.value
    if v and v not in cur:
        cur.append(v)
        via_list.options = cur
        via_list.value = v


def remove_via(_):
    cur = list(via_list.options)
    sel = via_list.value
    if sel in cur:
        cur.remove(sel)
        via_list.options = cur
        via_list.value = cur[-1] if cur else None


def move_up(_):
    cur = list(via_list.options)
    sel = via_list.value
    if sel is None:
        return
    i = cur.index(sel)
    if i > 0:
        cur[i-1], cur[i] = cur[i], cur[i-1]
        via_list.options = cur
        via_list.value = sel


def move_down(_):
    cur = list(via_list.options)
    sel = via_list.value
    if sel is None:
        return
    i = cur.index(sel)
    if i < len(cur)-1:
        cur[i+1], cur[i] = cur[i], cur[i+1]
        via_list.options = cur
        via_list.value = sel


def clear_vias(_):
    via_list.options = []
    via_list.value = None


def compute_preview(_):
    global preview_layer, preview_path_nodes, last_G, last_node_coords, last_seg_diag, last_waypoints

    with out:
        out.clear_output()
        progress.value = 0
        status.value = '<b>Computing preview...</b>'

        # remove old preview
        if preview_layer is not None:
            try:
                m.remove_layer(preview_layer)
            except Exception:
                pass
            preview_layer = None
            preview_path_nodes = None

        ordered = [start_dd.value] + list(via_list.options) + [end_dd.value]
        last_waypoints = ordered

        pts_xy = [(float(stations_lookup.loc[lab].geometry.x), float(stations_lookup.loc[lab].geometry.y)) for lab in ordered]

        mode = routing_mode_dd.value
        penalty = float(penalty_slider.value)
        do_node = bool(node_cb.value)

        G, node_coords, tree, node_to_comp, codes_used = build_graph(pts_xy, mode, penalty, node_intersections=do_node,
                                                                     progress=progress, status=status)

        print('Mode:', mode)
        if mode == PREFER_MRS:
            print('Penalty:', penalty)
        print('Node intersections:', do_node)
        print('Codes used:', codes_used)
        print('Waypoints:', ' → '.join(ordered))

        if tree is None or G.number_of_nodes() == 0:
            status.value = '<span style="color:red"><b>Graph empty.</b></span>'
            progress.value = 100
            return

        last_G = G
        last_node_coords = node_coords

        # snap
        snapped=[]
        for (x,y), lab in zip(pts_xy, ordered):
            d, n = tree.query([x,y], k=1)
            print(f"Snap {lab}: {d:.1f} m")
            snapped.append(int(n))

        seg_paths=[]
        seg_diag=[]

        for i in range(len(snapped)-1):
            s, t = snapped[i], snapped[i+1]
            seg = f"{ordered[i]} → {ordered[i+1]}"
            p = nx.shortest_path(G, s, t, weight='length_m') if mode == ALLOW_ALL else nx.shortest_path(G, s, t, weight='weight')
            km = path_length_m(G, p)/1000.0
            codes = codigo_breakdown_km(G, p)
            seg_diag.append({'segment': seg, 'km': km, 'codes': codes})
            seg_paths.append(p)

        last_seg_diag = seg_diag

        full = stitch_paths(seg_paths)
        preview_path_nodes = full

        coords = nodes_to_latlon(node_coords, full)
        preview_layer = Polyline(locations=coords, color=PREVIEW_COLOR, weight=PREVIEW_WEIGHT,
                                 opacity=PREVIEW_OPACITY, dash_array=PREVIEW_DASH,
                                 line_cap='butt', line_join='miter', name='preview')
        m.add_layer(preview_layer)

        clear_markers()
        for lab in ordered:
            add_marker(lab)

        status.value = '<span style="color:green"><b>Preview ready.</b></span> Commit Highlight to save.'
        progress.value = 100


def commit_highlight(_):
    global preview_path_nodes, last_node_coords
    with out:
        if preview_path_nodes is None or last_node_coords is None:
            print('No preview exists. Compute Preview first.')
            return

        nm = (highlight_name.value or '').strip() or 'highlight'
        # ensure unique
        if nm in highlight_layers:
            i=2
            while f"{nm} ({i})" in highlight_layers:
                i+=1
            nm = f"{nm} ({i})"

        coords = nodes_to_latlon(last_node_coords, preview_path_nodes)
        poly = Polyline(locations=coords, color=color_picker.value,
                        weight=HIGHLIGHT_WEIGHT, opacity=HIGHLIGHT_OPACITY,
                        line_cap=HIGHLIGHT_LINE_CAP, line_join=HIGHLIGHT_LINE_JOIN,
                        name=nm)
        m.add_layer(poly)
        highlight_layers[nm] = poly
        highlight_order.append(nm)
        status.value = f'<span style="color:green"><b>Committed highlight:</b></span> {nm}'
        print('Committed highlight:', nm)


def explain(_):
    with out:
        if not last_seg_diag:
            print('No diagnostics. Compute Preview first.')
            return
        print('=== EXPLAIN ===')
        print('Waypoints:', ' → '.join(last_waypoints or []))
        for d in last_seg_diag:
            print('Segment:', d['segment'])
            print(f"  length: {d['km']:.1f} km")
            print('  CodigoFerr breakdown (km):')
            for k,v in d['codes'].items():
                print(f"    {k}: {v:.1f} km")
        print('If length is still huge in Allow all, the corridor is disconnected in the geometry. Try keeping Node intersections ON and increasing SNAP_TOLERANCE_M / ROUND_M.')

btn_add_via.on_click(add_via)
btn_remove_via.on_click(remove_via)
btn_up.on_click(move_up)
btn_down.on_click(move_down)
btn_clear_vias.on_click(clear_vias)

btn_preview.on_click(compute_preview)
btn_commit.on_click(commit_highlight)
btn_explain.on_click(explain)

m


Map(center=[-22.114686312328335, -44.546276963592256], controls=(ZoomControl(options=['position', 'zoom_in_tex…